In [2]:
import pandas as pd
import duckdb
import os
import glob

In [3]:
#read files into dictionary
dfs = {}
for file in glob.glob(os.path.join("clean_data", "*.parquet")): #search for all parquet files in clean_data, and iterate
    name = os.path.basename(file).replace(".parquet", "")
    dfs[name] = pd.read_parquet(file)

In [4]:
# ANALYSIS 1: HAVE SPECIALISED MOLDS REPLACED ASSEMBLIES OF SMALLER PARTS TO CREATE THE SAME SHAPE? 
# ...OR VICE VERSA?

# THE 'PART_RELATIONSHIPS' TABLE SHOWS 'PARENT' PARTS THAT HAVE RELATED 'CHILD' PARTS.
# THESE 'CHILD' PARTS ARE EITHER SEGMENTS THAT COMBINE TO MAKE UP THE PARENT, 
# OR THEY ARE A COMINBATION OF MULTIPLE child PARTS.
# EITHER WAY, THE child(S) ARE THE PART(S) THAT WERE RELEASED FIRST.

# TABLES NEEDED (6 out of 12):

    # part_relationships - core table to identify child-child connections
    # parts - to get the part name, for human readability
    # part_categories - allow for more granular analysis by part category
    # inventory_parts, inventories, sets - the 'year' column in the 'sets' table is needed to get the timeline of parent and child part usage


#---note: the 'inventory_sets' table is not needed as it just records the quantity of a given part used per set, which is not needed here

In [13]:
part_relationships = dfs["part_relationships"]
parts = dfs["parts"]
part_categories = dfs["part_categories"]
inventory_parts = dfs["inventory_parts"]
inventories = dfs["inventories"]
inventory_sets = dfs["inventory_sets"]
sets = dfs["sets"]

In [14]:
#only look at the parts that have a parent-child relatiosnhip, not eg. a mold update or print
part_relationships = duckdb.sql("SELECT * FROM part_relationships WHERE rel_type = 'R'").df()

In [ ]:
#joined table is symmetrical in structure, where parent part info starts from the middle and continues through the left columns,
#and child part indo starts from the middle and continues through the right columnd
joined = duckdb.sql("""
                    
                    WITH parts_sets AS(
                        SELECT
                            p.part_num,
                            s.set_num,
                            s.year,
                            ivs.quantity
                        FROM parts p JOIN inventory_parts ip ON p.part_num = ip.part_num
                        JOIN inventories i ON ip.inventory_id = i.id
                        JOIN inventory_sets ivs ON i.set_num = ivs.set_num
                        JOIN sets s ON ivs.set_num = s.set_num
                    )

                    SELECT
                        
                        pc.name AS category,                         --only needs to be shows once, as parent and child likely to have same category 
                    
                        parent_ps.quantity AS parent_quantity,
                        parent_ps.year AS parent_year,
                        parent_ps.set_num AS parent_set_sum,
                        pr.parent_part_num,
                        parent_p.name AS parent_part_name,
                    
                        child_p.name AS child_part_name,
                        pr.child_part_num,
                        child_ps.set_num AS child_set_num,
                        parent_ps.year AS child_year,
                        parent_ps.quantity AS child_quantity
                    
                    FROM part_relationships pr
                    JOIN parts child_p ON pr.child_part_num = child_p.part_num
                    JOIN parts parent_p ON pr.parent_part_num = parent_p.part_num
                    JOIN part_categories pc ON child_p.part_cat_id = pc.id

                    JOIN parts_sets parent_ps ON parent_p.part_num = parent_ps.part_num
                    JOIN parts_sets child_ps ON child_p.part_num = child_ps.part_num

                    WHERE pr.rel_type = 'R'
                    


                    
                    """).df() #takes 3.2 - 3.3 seconds to executed

In [28]:
joined.head(100)

,category,parent_quantity,parent_year,parent_set_sum,parent_part_num,parent_part_name,child_part_name,child_part_num,child_set_num,child_year,child_quantity
0,Large Buildable Figures,1,2001,8531-1,32554,Large Figure Head Connector Block Eye/Brain Stalk,Large Figure Head Connector Block 3 x 4 x 1 2/3,32553,8541-1,2001,1
1,Wheels and Tyres,1,2003,4100-1,30391,Tyre 30.4 x 14 Offset Tread,Wheel 18 x 14 with Tread Small Hub,30285,8673-1,2003,1
2,"Supports, Girders and Cranes",1,2003,4100-1,30396,Hinge 1 x 2 Locking with 2 Fingers and Towball...,Hook with Towball,30395,7642-1,2003,1
3,Wheels and Tyres,1,2003,4100-1,30648,Tyre 24 x 14 Shallow Tread (Tread Small Hub),"Wheel 18 x 14 with Pin Hole, Fake Bolts and Sh...",55981,60080-1,2003,1
4,"Hinges, Arms and Turntables",1,2003,4100-1,3680,"Turntable 2 x 2 Plate, Base",Turntable 2 x 2 Plate - Top,3679,6241-1,2003,1
...,...,...,...,...,...,...,...,...,...,...,...
95,Containers,1,2002,5850-1,4739a,Treasure Chest Lid [Thick Hinge],"Treasure Chest Bottom with Rear Slots, No Groove",4738c,70738-1,2002,1
96,Containers,1,2002,5850-1,4740,Dish 2 x 2 Inverted [Radar],Trash Can with 2 Cover Holders,2439,7994-1,2002,1
97,Bricks Round and Cones,1,2002,5850-1,6162,Brick Round Corner 12 x 12,Brick Special 24 x 24 without 12 x 12 Quarter ...,6161,5807-1,2002,1
98,Wheels and Tyres,1,2009,9686-1,2815,Technic Wedge Belt Wheel Tire,Technic Wedge Belt Wheel [aka Pulley],4185,7642-1,2009,1
